In [ ]:
# Importamos Pytorch
import torch

print("Version de PyTorch:", torch.__version__)
print("GPU disponible:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("Nombre de GPU:", torch.cuda.get_device_name(0))

    memoria = torch.cuda.get_device_properties(0).total_memory
    print("Memoria GPU:", round(memoria / (1024**3), 2), "GB")

else:
    print("No se asignó una GPU CUDA.")


In [ ]:

%pip install -q -U diffusers transformers accelerate safetensors


In [ ]:

import torch
import diffusers
import transformers
import accelerate

print("PyTorch:", torch.__version__)
print("Diffusers:", diffusers.__version__)
print("Transformers:", transformers.__version__)
print("Accelerate:", accelerate.__version__)

print("CUDA disponible:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
# Verificamos que PyTorch reconozca la GPU

import torch

print("Versión de PyTorch:", torch.__version__)
print("GPU disponible:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("ATENCIÓN: No hay una GPU disponible.")

In [ ]:
# Probemos cargar Stable Diffusion
import torch
from diffusers import StableDiffusionUpscalePipeline

modelo = "stabilityai/stable-diffusion-x4-upscaler"

# Liberar memoria de GPU
torch.cuda.empty_cache()

print("Cargando Stable Diffusion x4 Upscaler...")

pipeline = StableDiffusionUpscalePipeline.from_pretrained(
    modelo,
    torch_dtype=torch.float16,
    use_safetensors=True
)

# Administrar memoria entre CPU y GPU
pipeline.enable_model_cpu_offload()

print("================================")
print("MODELO CARGADO CORRECTAMENTE")
print("================================")
print("Modelo:", modelo)
print("GPU:", torch.cuda.get_device_name(0))


## Prueba de funcionamiento de Stable Diffusion

Celda utilizada para comprobar la carga y el funcionamiento
del modelo. No forma parte del procesamiento de la aplicación.

In [ ]:
## Prueba de funcionamiento de Stable Diffusion

import requests
from PIL import Image
from io import BytesIO
from IPython.display import display

# Descargar imagen de prueba oficial
url = "https://huggingface.co/datasets/hf-internal-testing/diffusers-images/resolve/main/sd2-upscale/low_res_cat.png"

response = requests.get(url, timeout=30)
response.raise_for_status()

imagen_original = Image.open(
    BytesIO(response.content)
).convert("RGB")

# Reducir la imagen para la primera prueba
imagen_original = imagen_original.resize((128, 128))

print("Imagen original:", imagen_original.size)

# Ejecutar el modelo
resultado = pipeline(
    prompt="a white cat, realistic photograph",
    image=imagen_original,
    num_inference_steps=10,
    noise_level=20,
    guidance_scale=7.5
).images[0]

print("Imagen restaurada:", resultado.size)

display(imagen_original)
display(resultado)

resultado.save("/content/imagen_ampliada.png")

print("Imagen guardada correctamente.")


In [ ]:
# Cargamos DETR

import torch
from transformers import pipeline as hf_pipeline

# Cargar el modelo de detección de objetos
detector = hf_pipeline(
    "object-detection",
    model="facebook/detr-resnet-50",
    device=0 if torch.cuda.is_available() else -1
)

print("MODELO DETR CARGADO CORRECTAMENTE")

In [ ]:
# Analizar la imagen original
detecciones_original = detector(imagen_original)

# Mostrar los objetos detectados
for objeto in detecciones_original:
    print(
        "Objeto:", objeto["label"],
        "| Confianza:", round(objeto["score"], 3)
    )

In [ ]:
# Analizar la imagen ampliada
detecciones_mejorada = detector(resultado)

# Mostrar los objetos detectados
for objeto in detecciones_mejorada:
    print(
        "Objeto:", objeto["label"],
        "| Confianza:", round(objeto["score"], 3)
    )

In [ ]:
import sys
import torch

print("Versión de Python:", sys.version)
print("Versión de PyTorch:", torch.__version__)
print("GPU disponible:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
try:
    import gradio as gr
    print("Gradio instalado:", gr.__version__)
except ImportError:
    print("Gradio todavía no está instalado")

In [ ]:
# Definimos las funciones preparar_imagen, describir_detecciones y procesar_fotografia

from PIL import Image
import torch
import gradio as gr


def preparar_imagen(imagen, max_lado=256):
    """
    Reduce la imagen manteniendo su proporción.
    Ajusta las dimensiones a múltiplos de 8.
    """

    ancho, alto = imagen.size

    escala = min(max_lado / max(ancho, alto), 1)

    nuevo_ancho = max(8, round((ancho * escala) / 8) * 8)
    nuevo_alto = max(8, round((alto * escala) / 8) * 8)

    return imagen.resize(
        (nuevo_ancho, nuevo_alto),
        Image.Resampling.LANCZOS
    )


def describir_detecciones(imagen, detector):
    """
    Detecta objetos y devuelve sus etiquetas y puntuaciones.
    """

    resultados = detector(imagen)

    if not resultados:
        return "No se detectaron objetos."

    lineas = []

    for objeto in resultados[:5]:
        etiqueta = objeto["label"]
        confianza = objeto["score"]

        lineas.append(
            f"- {etiqueta}: {confianza:.2f}"
        )

    return "\n".join(lineas)



def procesar_fotografia(imagen, semilla, nivel_reconstruccion):
    """
    Reconstruye una fotografía con difusión
    y analiza su contenido con DETR.
    """

    if imagen is None:
        raise gr.Error("Primero tenés que cargar una fotografía.")

    if semilla is None or not float(semilla).is_integer():
        raise gr.Error("Ingresá un número entero como semilla.")

    semilla = int(semilla)

    imagen_original = imagen.convert("RGB")
    imagen_entrada = preparar_imagen(imagen_original)

    generador = torch.Generator(
        device="cuda"
    ).manual_seed(semilla)

    imagen_reconstruida = pipeline(
        prompt="A realistic, detailed photograph. Preserve the original subject, appearance and composition. Natural colors and realistic textures.",
        image=imagen_entrada,
        num_inference_steps=30,
        noise_level=int(nivel_reconstruccion),
        guidance_scale=3,
        generator=generador
    ).images[0]

    detecciones_original = describir_detecciones(
        imagen_original, detector
    )

    detecciones_reconstruida = describir_detecciones(
        imagen_reconstruida, detector
    )

    analisis = (
        f"SEMILLA UTILIZADA: {semilla}\n\n"
        f"NIVEL DE RECONSTRUCCIÓN: {nivel_reconstruccion}\n\n"
        f"OBJETOS EN LA FOTOGRAFÍA ORIGINAL:\n"
        f"{detecciones_original}\n\n"
        f"OBJETOS EN LA FOTOGRAFÍA RECONSTRUIDA:\n"
        f"{detecciones_reconstruida}\n\n"
        "IMPORTANTE: los detalles generados por difusión "
        "pueden no coincidir con la fotografía original."
    )

    return imagen_original, imagen_reconstruida, analisis


## Servicio Gradio - Backend de procesamiento

In [ ]:


with gr.Blocks(title="Reviví tus fotos") as demo:

    gr.Markdown("# Reviví tus fotos")
    gr.Markdown(
        "Servicio de restauración y reconstrucción "
        "de fotografías mediante inteligencia artificial."
    )

    imagen = gr.Image(
        label="Fotografía original",
        type="pil"
    )

    semilla = gr.Number(
        label="Semilla",
        value=42,
        precision=0
    )

    nivel_reconstruccion = gr.Radio(
        choices=[
            ("Conservador", 1),
            ("Equilibrado", 20),
            ("Creativo", 50)
        ],
        value=1,
        label="Nivel de reconstrucción",
        info=(
            "Elegí cuánto puede variar la imagen "
            "durante la reconstrucción."
        )
    )

    boton = gr.Button("Procesar fotografía")

    original = gr.Image(
        label="Fotografía original"
    )

    reconstruida = gr.Image(
        label="Fotografía reconstruida"
    )

    analisis = gr.Textbox(
        label="Análisis de la fotografía",
        lines=8,
        interactive=False
    )

    boton.click(
        fn=procesar_fotografia,
        inputs=[
            imagen,
            semilla,
            nivel_reconstruccion
        ],
        outputs=[
            original,
            reconstruida,
            analisis
        ],
        api_name="procesar"
    )


demo.launch(share=True, debug=True)


## Diagnóstico de fotografías originales

In [ ]:
from PIL import Image
from google.colab import files

archivos = files.upload()

for nombre in archivos:
    imagen = Image.open(nombre)

    print("Archivo:", nombre)
    print("Dimensiones:", imagen.size)
    print("Formato:", imagen.format)
    print("Peso:", round(len(archivos[nombre]) / 1024), "KB")
    print("-------------------------")


## Registro de experimentación

### Configuración actual
- Modelo: stabilityai/stable-diffusion-x4-upscaler
- Tamaño máximo de entrada: 256 px
- Semilla: 42
- Pasos de inferencia: 30
- Noise level: 1
- Guidance scale: 3

### Resultados obtenidos

**Caso 1: chica con buzo verde**
- Mejora visual al aumentar la entrada de 128 a 256 px.
- Persisten modificaciones en la apariencia original.

**Caso 2: fotografía grupal de cumpleaños**
- No se recuperaron significativamente los rostros desenfocados.

**Caso 3: gato (176 x 209 px)**
- Resultado visual favorable.
- Mayor definición y conservación aparente de la imagen original.

### Próximo experimento
Procesar una segunda fotografía de un gato o perro de dimensiones similares, manteniendo los parámetros actuales.
